In [ ]:
import json
from datetime import datetime, timedelta

import psplib  # pip install psplib


def to_iso(dt: datetime) -> str:
    return dt.strftime("%Y-%m-%dT%H:%M:%S")


def convert_psplib_sm_to_outage_json(
    sm_path: str,
    out_path: str,
    outage_id: str = "PSPLIB_BENCH",
    start_date: str = "2026-01-01",
    working_hours_per_day: int = 24,
    mode_index: int = 0,  # for multi-mode instances you can pick which mode to export
):
    inst = psplib.parse(sm_path, instance_format="psplib")

    # Activities and resources in this library:
    # inst.activities[i].modes[m].duration, .demands
    # inst.activities[i].successors -> indices (0-based)
    # inst.resources[r].capacity
    n_jobs = inst.num_activities
    n_res = inst.num_resources

    # Choose a mode per activity (RCPSP single-mode: mode_index=0 is correct)
    durations = []
    demands = []
    successors = []
    for i, act in enumerate(inst.activities):
        if mode_index >= len(act.modes):
            raise ValueError(
                f"Activity {i+1} has only {len(act.modes)} modes; cannot use mode_index={mode_index}."
            )
        m = act.modes[mode_index]
        durations.append(int(m.duration))
        demands.append([int(x) for x in m.demands])
        successors.append(list(act.successors))  # already 0-based indices per parser

    capacities = [int(r.capacity) for r in inst.resources]

    # A simple horizon upper bound: sum of durations
    horizon_units = sum(durations)

    start_dt = datetime.fromisoformat(start_date)
    # Interpret 1 time unit as (24 / working_hours_per_day) hours in calendar time
    hours_per_unit = 24.0 / float(working_hours_per_day)
    end_dt = start_dt + timedelta(hours=horizon_units * hours_per_unit)

    # Build tasks
    tasks = []
    for i in range(n_jobs):
        task_id = f"J{i+1}"
        succ_ids = [f"J{j+1}" for j in successors[i]]

        req_resources = []
        for r in range(n_res):
            d = demands[i][r]
            if d > 0:
                req_resources.append({"skill_type": f"R{r+1}", "crew_count": d})

        tasks.append(
            {
                "task_id": task_id,
                "description": f"Activity {i+1}",
                "duration": float(durations[i]),
                "successors": succ_ids,
                "location_id": None,
                "required_resources": req_resources,
                "required_equipment": [],
                "is_hold_point": False,
            }
        )

    # Build resources with constant availability
    resources = []
    for r in range(n_res):
        resources.append(
            {
                "skill_type": f"R{r+1}",
                "availability_periods": [
                    {
                        "start_date": to_iso(start_dt),
                        "end_date": to_iso(end_dt),
                        "available_count": capacities[r],
                        "reason": "PSPLIB constant capacity",
                    }
                ],
            }
        )

    data = {
        "outage": {
            "outage_id": outage_id,
            "start_date": start_dt.date().isoformat(),
            "target_end_date": None,
            "working_hours_per_day": working_hours_per_day,
        },
        "tasks": tasks,
        "resources": resources,
        "equipment": [],
        "locations": [],
    }

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)


In [4]:
convert_psplib_sm_to_outage_json(sm_path="benchmarks/j301_1.sm",
                                 out_path="j301_1.json")

AttributeError: 'ProjectInstance' object has no attribute 'durations'